In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("data.csv")

In [4]:
df.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [5]:
df["Car"] = df["Make"].astype(str) + " " + df["Model"].astype(str)

In [7]:
# Count cars before grouping
car_counts = df["Car"].value_counts()

# Check rare cars
print("Total cars:", df["Car"].nunique())

print("Cars appearing once:",
      (car_counts == 1).sum())

print("Cars appearing less than 10 times:",
      (car_counts < 10).sum())

print("Cars appearing at least 10 times:",
      (car_counts >= 10).sum())

Total cars: 928
Cars appearing once: 41
Cars appearing less than 10 times: 518
Cars appearing at least 10 times: 410


In [8]:
car_counts = df["Car"].value_counts()

for threshold in [2, 5, 10]:

    selected_cars = car_counts[car_counts >= threshold]

    selected_rows = df["Car"].isin(selected_cars.index).sum()

    print(f"\nMinimum samples: {threshold}")
    print("Number of car classes:", len(selected_cars))
    print("Rows retained:", selected_rows)
    print("Rows removed:", len(df) - selected_rows)


Minimum samples: 2
Number of car classes: 887
Rows retained: 11873
Rows removed: 41

Minimum samples: 5
Number of car classes: 665
Rows retained: 11219
Rows removed: 695

Minimum samples: 10
Number of car classes: 410
Rows retained: 9464
Rows removed: 2450


In [9]:
# Minimum samples per car
MIN_SAMPLES = 5

# Count occurrences
car_counts = df["Car"].value_counts()

# Keep cars with at least 5 records
df = df[
    df["Car"].map(car_counts) >= MIN_SAMPLES
].copy()

print("Remaining rows:", len(df))
print("Remaining car classes:", df["Car"].nunique())

Remaining rows: 11219
Remaining car classes: 665


In [11]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [12]:
numeric_features = [
    "MSRP",
    "Year",
    "Number of Doors",
    "Engine Cylinders",
    "Engine HP"
]

categorical_features = [
    "Engine Fuel Type",
    "Transmission Type",
    "Driven_Wheels",
    "Vehicle Size",
    "Vehicle Style"
]

X = df[numeric_features + categorical_features]
y = df["Car"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (8975, 10)
Test: (2244, 10)


In [13]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=35,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=35,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=25,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=7,
        weights="distance",
        metric="minkowski",
        p=2
    )
}

In [14]:
from sklearn.metrics import accuracy_score, top_k_accuracy_score

results = []
fitted_models = {}

for name, clf in models.items():

    print(f"\nTraining {name}...")

    pipe = Pipeline([
        ("prep", preprocess),
        ("clf", clf)
    ])

    pipe.fit(X_train, y_train)

    train_pred = pipe.predict(X_train)

    test_pred = pipe.predict(X_test)

    probabilities = pipe.predict_proba(X_test)

    classes = pipe.classes_

    train_acc = accuracy_score(
        y_train,
        train_pred
    )

    top1 = accuracy_score(
        y_test,
        test_pred
    )

    top5 = top_k_accuracy_score(
        y_test,
        probabilities,
        k=5,
        labels=classes
    )

    results.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Top-1 Accuracy": top1,
        "Top-5 Accuracy": top5,
        "Overfitting Gap": train_acc - top1
    })

    fitted_models[name] = pipe


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Top-1 Accuracy",
    ascending=False
)

print("\n========== MODEL COMPARISON ==========")

print(results_df.round(4).to_string(index=False))


Training Random Forest...

Training Extra Trees...

Training Decision Tree...

Training KNN...

========== MODEL COMPARISON ==========
        Model  Train Accuracy  Top-1 Accuracy  Top-5 Accuracy  Overfitting Gap
Random Forest          0.9440          0.7825          0.9875           0.1614
  Extra Trees          0.9294          0.7611          0.9808           0.1682
          KNN          0.9914          0.7206          0.9483           0.2708
Decision Tree          0.7062          0.6199          0.7995           0.0863


In [15]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

rf_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "clf__n_estimators": [200, 300, 500],

    "clf__max_depth": [15, 20, 25, 30, None],

    "clf__min_samples_split": [4, 6, 8, 10],

    "clf__min_samples_leaf": [2, 3, 4, 5],

    "clf__max_features": ["sqrt", 0.5, 0.7],

    "clf__class_weight": [None, "balanced_subsample"]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=15,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV Accuracy:")
print(random_search.best_score_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best Parameters:
{'clf__n_estimators': 300, 'clf__min_samples_split': 6, 'clf__min_samples_leaf': 2, 'clf__max_features': 0.7, 'clf__max_depth': 20, 'clf__class_weight': None}

Best CV Accuracy:
0.8054587857452332


In [16]:
best_rf = random_search.best_estimator_

train_predictions = best_rf.predict(X_train)

test_predictions = best_rf.predict(X_test)

train_accuracy = accuracy_score(
    y_train,
    train_predictions
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

print("Train Accuracy:", round(train_accuracy, 4))

print("Test Top-1 Accuracy:", round(test_accuracy, 4))

print(
    "Overfitting Gap:",
    round(train_accuracy - test_accuracy, 4)
)

Train Accuracy: 0.946
Test Top-1 Accuracy: 0.8365
Overfitting Gap: 0.1095


In [17]:
probabilities = best_rf.predict_proba(X_test)

top5 = top_k_accuracy_score(
    y_test,
    probabilities,
    k=5,
    labels=best_rf.classes_
)

print("Top-5 Accuracy:", round(top5, 4))

Top-5 Accuracy: 0.9955


In [19]:
feature_names = best_rf.named_steps["prep"].get_feature_names_out()

importances = best_rf.named_steps["clf"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.head(20))

                                              Feature  Importance
4                                      num__Engine HP    0.346619
0                                           num__MSRP    0.170594
1                                           num__Year    0.141283
3                               num__Engine Cylinders    0.049393
14             cat__Engine Fuel Type_regular unleaded    0.018049
2                                num__Number of Doors    0.017306
22               cat__Driven_Wheels_front wheel drive    0.017187
41                           cat__Vehicle Style_Sedan    0.016281
30                         cat__Vehicle Style_4dr SUV    0.015217
12  cat__Engine Fuel Type_premium unleaded (recomm...    0.014322
20                 cat__Driven_Wheels_all wheel drive    0.013351
16                   cat__Transmission Type_AUTOMATIC    0.013132
42                           cat__Vehicle Style_Wagon    0.012838
29                   cat__Vehicle Style_4dr Hatchback    0.012508
38        

In [22]:
!pip install catboost


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


xgboost

In [26]:
!pip install xgboost


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_lambda=5,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
])

xgb_pipeline.fit(
    X_train,
    y_train_encoded
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](665,)","[ 0, 1, 2,...,662,663,664]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['MSRP','Year','Number of Doors',...,'Driven_Wheels','Vehicle Size', 'Vehicle Style']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``rema

In [28]:
train_pred = xgb_pipeline.predict(X_train)

test_pred = xgb_pipeline.predict(X_test)

print(
    "Train Accuracy:",
    accuracy_score(y_train_encoded, train_pred)
)

print(
    "Test Top-1 Accuracy:",
    accuracy_score(y_test_encoded, test_pred)
)

Train Accuracy: 0.8683008356545961
Test Top-1 Accuracy: 0.7785204991087344


In [29]:
xgb_configs = {

    "XGB_1": {
        "n_estimators": 400,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "min_child_weight": 3,
        "reg_lambda": 5
    },

    "XGB_2": {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 10
    },

    "XGB_3": {
        "n_estimators": 400,
        "max_depth": 6,
        "learning_rate": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 5,
        "reg_lambda": 8
    }
}

In [30]:
from sklearn.model_selection import train_test_split

X_sub, X_val, y_sub, y_val = train_test_split(
    X_train,
    y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_train_encoded
)

results = []
tuned_models = {}

for name, params in xgb_configs.items():

    print(f"\nTraining {name}...")

    model = Pipeline([
        ("prep", preprocess),
        ("clf", XGBClassifier(
            **params,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
            tree_method="hist"
        ))
    ])

    model.fit(X_sub, y_sub)

    train_pred = model.predict(X_sub)
    val_pred = model.predict(X_val)

    train_acc = accuracy_score(y_sub, train_pred)
    val_acc = accuracy_score(y_val, val_pred)

    results.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Validation Accuracy": val_acc,
        "Gap": train_acc - val_acc
    })

    tuned_models[name] = model

tuning_results = pd.DataFrame(results)

print(
    tuning_results.sort_values(
        "Validation Accuracy",
        ascending=False
    ).round(4)
)


Training XGB_1...

Training XGB_2...

Training XGB_3...
   Model  Train Accuracy  Validation Accuracy     Gap
0  XGB_1          0.8801               0.7560  0.1241
1  XGB_2          0.7110               0.6479  0.0631
2  XGB_3          0.6638               0.6262  0.0376


In [31]:
best_config_name = (
    tuning_results
    .sort_values("Validation Accuracy", ascending=False)
    .iloc[0]["Model"]
)

best_xgb = tuned_models[best_config_name]

print("Best Configuration:", best_config_name)

Best Configuration: XGB_1


In [32]:
final_train_pred = best_xgb.predict(X_train)
final_test_pred = best_xgb.predict(X_test)

print(
    "Train Accuracy:",
    accuracy_score(y_train_encoded, final_train_pred)
)

print(
    "Test Top-1 Accuracy:",
    accuracy_score(y_test_encoded, final_test_pred)
)

Train Accuracy: 0.8552646239554318
Test Top-1 Accuracy: 0.7647058823529411
